# Day 4 — 技术面特征工程 (升级版)

**论文六大维度覆盖: 惯性 ✓ | 波动率/交易摩擦 ✓  

**修复/升级内容**:
- 原有10个特征保留并修正
- 新增: MACD, 布林带, ATR代理, 偏度/峰度, 更多动量周期, 历史最大回撤
- 目标: 从价格数据中尽可能提取信号

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

In [ ]:
# ==========================================
# 第1步: 读取Day3带标签数据
# ==========================================
df = pd.read_csv(r"C:\Users\1\Desktop\项目\stock-data\day3_dataset_mdd.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["symbol", "Date"]).reset_index(drop=True)

print("输入维度:", df.shape)
print("列:", df.columns.tolist())

In [ ]:
# ==========================================
# 第2步: 构建辅助函数
# ==========================================

def grouped_rolling(series, window, func_name):
    """groupby-aware rolling 操作"""
    return (
        series.groupby(df["symbol"])
        .rolling(window, min_periods=window)
        .agg(func_name)
        .reset_index(level=0, drop=True)
    )

def grouped_pct_change(series, periods):
    """groupby-aware pct_change"""
    return series.groupby(df["symbol"]).pct_change(periods)

def grouped_diff(series, periods=1):
    """groupby-aware diff"""
    return series.groupby(df["symbol"]).diff(periods)

In [ ]:
# ==========================================
# 第3步: 确保收益率字段
# ==========================================
if "return" not in df.columns or df["return"].isna().all():
    df["return"] = df.groupby("symbol")["close"].pct_change()

In [ ]:
# ==========================================
# 第4步: 动量类特征 (Inertia/Momentum) — 5个
# ==========================================

print("构造动量特征...")

# 多周期价格动量
for period in [5, 10, 20, 60, 120]:
    df[f"ret_{period}d"] = grouped_pct_change(df["close"], period)

# 收益率动量 (收益率的移动平均)
df["return_ma_5"] = grouped_rolling(df["return"], 5, "mean")
df["return_ma_20"] = grouped_rolling(df["return"], 20, "mean")

print("动量特征: 7个 (ret_5d/10d/20d/60d/120d + return_ma_5/20)")

In [ ]:
# ==========================================
# 第5步: 波动率类特征 — 4个
# ==========================================

print("构造波动率特征...")

# 历史波动率 (多周期)
for window in [5, 10, 20, 60]:
    df[f"vol_{window}d"] = grouped_rolling(df["return"], window, "std")

# 收益率偏度和峰度 (20日窗口)
df["skew_20d"] = grouped_rolling(df["return"], 20, "skew")
df["kurt_20d"] = grouped_rolling(df["return"], 20, lambda x: x.kurtosis())

print("波动率特征: 6个 (vol_5/10/20/60 + skew_20 + kurt_20)")

In [ ]:
# ==========================================
# 第6步: 趋势/均线类特征 — 6个
# ==========================================

print("构造趋势特征...")

# 移动平均
for w in [5, 10, 20, 60]:
    df[f"ma_{w}"] = grouped_rolling(df["close"], w, "mean")

# 价格-均线偏离
df["ma_gap_5"] = (df["close"] - df["ma_5"]) / df["ma_5"]
df["ma_gap_20"] = (df["close"] - df["ma_20"]) / df["ma_20"]

# 短期-长期均线差异 (趋势强度)
df["ma_trend_5_20"] = (df["ma_5"] - df["ma_20"]) / df["ma_20"]

print("趋势特征: 6个 (ma_5/10/20/60, ma_gap_5/20, ma_trend_5_20)")

In [ ]:
# ==========================================
# 第7步: MACD 特征 — 3个
# ==========================================

print("构造MACD特征...")

# EMA(12) 和 EMA(26)
df["ema_12"] = grouped_rolling(df["close"], 12, lambda x: x.ewm(span=12, adjust=False).mean().iloc[-1])
df["ema_26"] = grouped_rolling(df["close"], 26, lambda x: x.ewm(span=26, adjust=False).mean().iloc[-1])

# 改用SMA近似EMA (避免rolling内lambda性能问题)
# EMA(12) 平滑因子 = 2/(12+1) = 0.1538
def ema(series, span):
    """按group计算EMA"""
    return series.groupby(df["symbol"]).transform(lambda x: x.ewm(span=span, adjust=False).mean())

df["ema_12"] = ema(df["close"], 12)
df["ema_26"] = ema(df["close"], 26)

# DIF = EMA(12) - EMA(26)
df["macd_dif"] = df["ema_12"] - df["ema_26"]

# DEA = EMA(DIF, 9)
df["macd_dea"] = ema(df["macd_dif"], 9)

# MACD柱 = 2 * (DIF - DEA)
df["macd_hist"] = 2 * (df["macd_dif"] - df["macd_dea"])

print("MACD特征: 3个 (macd_dif, macd_dea, macd_hist)")

In [ ]:
# ==========================================
# 第8步: 布林带特征 — 3个
# ==========================================

print("构造布林带特征...")

df["bb_upper"] = df["ma_20"] + 2 * df["vol_20d"] * df["close"]
df["bb_lower"] = df["ma_20"] - 2 * df["vol_20d"] * df["close"]

# %B = 价格在带中的位置 (0=下轨, 1=上轨)
df["bb_pct_b"] = (df["close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"] + 1e-10)

# 带宽 = (上轨-下轨) / 中轨
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / (df["ma_20"] + 1e-10)

print("布林带特征: 3个 (bb_pct_b, bb_width)")

In [ ]:
# ==========================================
# 第9步: ATR代理 (基于收盘价波动) — 2个
# 注: 理想情况需要OHLC数据, 此处用收盘价变化幅度代理
# ==========================================

print("构造ATR代理...")

# 价格变化绝对值作为波动代理
df["close_change_abs"] = grouped_diff(df["close"]).abs()

# ATR(14) 代理 = 14日平均绝对变化
df["atr_14"] = grouped_rolling(df["close_change_abs"], 14, "mean")

# 归一化ATR
df["atr_pct"] = df["atr_14"] / (df["close"] + 1e-10)

print("ATR特征: 2个 (atr_14, atr_pct)")

In [ ]:
# ==========================================
# 第10步: 回撤特征 — 3个
# ==========================================

print("构造回撤特征...")

# 滚动N日最高价
for w in [20, 60]:
    roll_max = grouped_rolling(df["close"], w, "max")
    df[f"drawdown_{w}d"] = (df["close"] - roll_max) / roll_max

# 是否处于回撤状态
df["is_drawdown_20"] = (df["drawdown_20d"] < 0).astype(int)

print("回撤特征: 3个 (drawdown_20d/60d, is_drawdown_20)")

In [ ]:
# ==========================================
# 第11步: RSI 特征 — 1个
# ==========================================

print("构造RSI特征...")

delta = df.groupby("symbol")["close"].diff()
gain = delta.clip(lower=0)
loss = (-delta).clip(lower=0)

avg_gain = gain.groupby(df["symbol"]).rolling(14, min_periods=14).mean().reset_index(level=0, drop=True)
avg_loss = loss.groupby(df["symbol"]).rolling(14, min_periods=14).mean().reset_index(level=0, drop=True)

rs = avg_gain / (avg_loss + 1e-10)
df["rsi_14"] = 100 - (100 / (1 + rs))

print("RSI特征: 1个 (rsi_14)")

In [ ]:
# ==========================================
# 第12步: Z-score 特征 — 1个
# ==========================================

print("构造Z-score特征...")

roll_mean_20 = grouped_rolling(df["close"], 20, "mean")
roll_std_20 = grouped_rolling(df["close"], 20, "std")
df["zscore_20"] = (df["close"] - roll_mean_20) / (roll_std_20 + 1e-10)

print("Z-score特征: 1个")

In [ ]:
# ==========================================
# 第13步: 收益率离散特征 — 2个
# ==========================================

print("构造离散特征...")

# 连续上涨/下跌天数 (基于收益率符号)
return_sign = df.groupby("symbol")["return"].transform(lambda x: np.sign(x.fillna(0)))

# 简化: 正收益天数占比 (20日)
df["up_days_ratio_20"] = grouped_rolling((df["return"] > 0).astype(float), 20, "mean")

# 最大单日涨幅/跌幅 (20日)
df["max_return_20d"] = grouped_rolling(df["return"], 20, "max")
df["min_return_20d"] = grouped_rolling(df["return"], 20, "min")

print("离散特征: 3个 (up_days_ratio_20, max_return_20, min_return_20)")

In [ ]:
# ==========================================
# 第14步: 删除NaN并检查
# ==========================================

print(f"删除NaN前: {df.shape}")
df = df.dropna().reset_index(drop=True)
print(f"删除NaN后: {df.shape}")

# 确保crash列是整数
for c in ["crash", "crash_10", "crash_15", "crash_20"]:
    if c in df.columns:
        df[c] = df[c].astype(int)

In [ ]:
# ==========================================
# 第15步: 特征清单
# ==========================================

feature_cols = [c for c in df.columns if c not in 
               ["Date", "symbol", "close", "return", "future_mdd_20",
                "crash", "crash_10", "crash_15", "crash_20",
                "bb_upper", "bb_lower", "ema_12", "ema_26", "close_change_abs"]]

print(f"特征数: {len(feature_cols)}")
print("\n特征列表:")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {f}")

print(f"\n标签分布:")
print(df["crash"].value_counts())
print(f"Crash比例: {df['crash'].mean():.2%}")

In [ ]:
# ==========================================
# 第16步: 保存
# ==========================================

# 保存完整特征集
output_cols = ["Date", "symbol", "close", "return", "future_mdd_20",
               "crash", "crash_10", "crash_15", "crash_20"] + feature_cols

df[output_cols].to_csv(
    r"C:\Users\1\Desktop\项目\stock-data\day4_features.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Day4 完成, 已保存 day4_features.csv")
print(f"最终维度: {df[output_cols].shape}")